# Neural reduced GP

AutoEncoder / VAE / supervised AutoEncoder を軽量設定で構築し、非線形 latent representation の使い方を確認します。


## データ


In [ ]:
import torch
from robotorchan.models import AutoEncoderGP, VAEGP, SupervisedAutoEncoderGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.rand(20, 6)
Y = torch.sin(4 * X[:, :1]) + X[:, 1:2]


## 軽量なreducer学習


In [ ]:
models = {
    "AE": AutoEncoderGP(X, Y, latent_dim=2, hidden_dims=(8,), epochs=3),
    "VAE": VAEGP(X, Y, latent_dim=2, hidden_dims=(8,), epochs=3),
    "SupervisedAE": SupervisedAutoEncoderGP(X, Y, latent_dim=2, hidden_dims=(8,), epochs=3),
}
for name, model in models.items():
    print(name, model.reduced_input_dim)


## Posterior


In [ ]:
X_test = torch.rand(2, 6)
for name, model in models.items():
    print(name, model.posterior(X_test).mean.squeeze(-1))


## Model fitting

Reducerは構築時に学習済みですが、GP部分は既存の`make_mll()`で別途学習します。


In [ ]:
from botorch.fit import fit_gpytorch_mll

for name, model in models.items():
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    print(name, "GP fitted")


## Acquisition-based candidate generation and evaluation

各モデルについて元の6次元空間でqLogEIを最適化し、提案点を目的関数で評価します。


In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.sampling.normal import SobolQMCNormalSampler


def objective(X):
    return torch.sin(4 * X[..., :1]) + X[..., 1:2]


bounds = torch.stack([torch.zeros(6), torch.ones(6)])
for name, model in models.items():
    acqf = qLogExpectedImprovement(
        model=model,
        best_f=Y.max(),
        sampler=SobolQMCNormalSampler(sample_shape=torch.Size([32])),
    )
    candidate, value = optimize_acqf(
        acq_function=acqf,
        bounds=bounds,
        q=1,
        num_restarts=2,
        raw_samples=32,
    )
    assert candidate.shape == torch.Size([1, 6])
    print(name, "candidate:", candidate)
    print(name, "acquisition:", value)
    print(name, "objective:", objective(candidate))


## Jointly trained neural encoders

`JointEncoderGP` と `HybridAutoEncoderGP` は、reducerを事前学習して固定するモデルとは異なり、GPの損失とencoderを同時に最適化します。Hybridでは再構成損失も含むため、既存の `training_loss()` を用います。


In [ ]:
from robotorchan.models import HybridAutoEncoderGP, JointEncoderGP

joint_models = {
    "JointEncoderGP": JointEncoderGP(X, Y, latent_dim=2, hidden_dims=(8,)),
    "HybridAutoEncoderGP": HybridAutoEncoderGP(
        X, Y, latent_dim=2, hidden_dims=(8,), reconstruction_weight=0.1
    ),
}
for name, model in joint_models.items():
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(5):
        optimizer.zero_grad()
        loss = model.training_loss()
        loss.backward()
        optimizer.step()
    model.eval()
    model.likelihood.eval()
    with torch.no_grad():
        pool = torch.rand(32, X.shape[-1])
        scores = model.posterior(pool).mean.reshape(-1)
    candidate = pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, X.shape[-1])
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate))


## Joint VAE and mixed joint encoder models

VAEではGPの予測損失に再構成・KL正則化を加えます。Mixedモデルは連続特徴量だけをencoderに通し、カテゴリ特徴量を保持します。


In [ ]:
from robotorchan.models import JointVAEGP, MixedJointEncoderGP, MixedJointVAEGP

mixed_X = torch.cat(
    [X, torch.randint(0, 2, (X.shape[0], 1)).to(X)], dim=-1
)
mixed_pool = torch.cat(
    [torch.rand(32, X.shape[-1]), torch.randint(0, 2, (32, 1)).to(X)],
    dim=-1,
)
additional_joint_models = {
    "JointVAEGP": (JointVAEGP(
        X, Y, latent_dim=2, hidden_dims=(8,), beta=0.1,
        reconstruction_weight=0.1,
    ), torch.rand(32, X.shape[-1])),
    "MixedJointEncoderGP": (MixedJointEncoderGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,),
    ), mixed_pool),
    "MixedJointVAEGP": (MixedJointVAEGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,),
        beta=0.1, reconstruction_weight=0.1,
    ), mixed_pool),
}
for name, (model, pool) in additional_joint_models.items():
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    for _ in range(5):
        optimizer.zero_grad()
        loss = model.training_loss()
        loss.backward()
        optimizer.step()
    model.eval()
    model.likelihood.eval()
    with torch.no_grad():
        scores = model.posterior(pool).mean.reshape(-1)
    candidate = pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape[-1] == pool.shape[-1]
    if pool.shape[-1] == 7:
        assert candidate[0, -1].item() in (0, 1)
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate[..., :6]))


## Frozen mixed neural input reducers

Mixed AutoEncoderとMixed VAEでは連続入力だけを事前学習したreducerで変換し、カテゴリ変数は保持します。GP部分は既存の `make_mll()` を使って学習します。


In [ ]:
from robotorchan.models import MixedAutoEncoderGP, MixedVAEGP

frozen_mixed_models = {
    "MixedAutoEncoderGP": MixedAutoEncoderGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,), epochs=3
    ),
    "MixedVAEGP": MixedVAEGP(
        mixed_X, Y, latent_dim=2, cat_dims=[6], hidden_dims=(8,), epochs=3
    ),
}
for name, model in frozen_mixed_models.items():
    fit_gpytorch_mll(model.make_mll())
    model.eval()
    with torch.no_grad():
        scores = model.posterior(mixed_pool).mean.reshape(-1)
    candidate = mixed_pool[scores.argmax()].unsqueeze(0)
    assert candidate.shape == (1, 7)
    assert candidate[0, -1].item() in (0, 1)
    print(name, "candidate:", candidate)
    print(name, "objective:", objective(candidate[..., :6]))
